# Project 4 — Stochasticity calibration on CV (fiducial-parameter test)

**Goal.** Test whether BIND2's generative variance is *physically* calibrated.
Diagonals (variances) can be matched by tuning noise scale; off-diagonals
(e.g. Cov[$M_\star$, $M_{\rm gas}$]) require the model to know that star
formation depletes gas — a relationship a pure interpolator does not learn.

**Design.**  CV gives ~1500 halos at the fiducial parameters from 27
independent ICs.  We use those as the truth distribution.  For a
stratified subset of 50 halos (5 logM bins × 10 halos) we generate $K{=}64$
BIND samples per halo, and then run a *law-of-total-variance* check:

$$\Sigma^{\rm truth}_{\rm halos} \;\overset{?}{=}\;
\underbrace{\Sigma^{\rm BIND}_{\rm across-halo, K=1}}_\text{epistemic / DMO-driven}
\;+\;
\underbrace{\langle \Sigma^{\rm BIND}_{\rm intra-halo} \rangle_h}_\text{aleatoric / generative}.$$

We work in *mass-residual* / *fraction* space (so M$_{200c}$ variation does
not dominate the off-diagonals at fixed mass).

**Pre-registered tests** (see Section 7):
1. 1D KS distance for $f_\star$ truth vs BIND across-halo, $<0.1$ within each mass bin
2. Var[$f_\star$] truth vs BIND total within factor 2 in each mass bin
3. Sign of Corr[$f_\star$, $f_{\rm gas}$] matches truth and within $\pm 0.3$
4. Sign of Corr[$q_{\rm DM}/q_{\rm DMO}$, $M_\star$] matches truth

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

ROOT = Path('/mnt/home/mlee1/vdm_bind2')
sys.path.insert(0, str(ROOT))
from metrics import CHANNEL_NAMES, radial_profile
from data import NormStats
from train import FlowMatchingLit
from test_suite.pipeline import normalize_cutout, _denormalize_to_physical

plt.rcParams.update({
    'font.size': 10, 'font.family': 'serif', 'mathtext.fontset': 'cm',
    'figure.dpi': 110, 'savefig.bbox': 'tight',
})

SUITE_ROOT = Path('/mnt/home/mlee1/ceph/fm_testsuite')
RUN_DIR    = Path('/mnt/home/mlee1/ceph/fm_runs/fm_two_head')
CKPT       = RUN_DIR / 'checkpoints' / 'last.ckpt'
NORM_STATS = RUN_DIR / 'norm_stats.npz'

SNAP, MASS_TAG, MODEL_NAME = 'snap_090', 'mass_threshold_1p000e13', 'fm_two_head'
BOX_SIZE, N_PIX_FULL, PATCH_PIX = 50.0, 1024, 128
MPC_PER_PIX = (BOX_SIZE * PATCH_PIX / N_PIX_FULL) / PATCH_PIX
N_RBINS = 32

CACHE_DIR = ROOT / 'analysis_physics_cache'
CACHE_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / 'paper_figures'
FIG_DIR.mkdir(exist_ok=True)

BASE_CACHE = CACHE_DIR / f'halo_features_{MODEL_NAME}.npz'
PROJ4_CACHE = CACHE_DIR / f'proj4_stochasticity_{MODEL_NAME}.npz'

# Selection / generation knobs
N_BINS_M    = 5
HALOS_PER_BIN = 10
K_SAMPLES   = 64
N_STEPS     = 50
BATCH_SIZE  = 16
USE_AMP     = True
RNG_SEED    = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Checkpoint:', CKPT)

## 1. Load truth features for CV halos

Reuse the existing per-halo cache for the K=1 baseline and the truth distribution.

In [ ]:
_base = np.load(BASE_CACHE, allow_pickle=False)
_cv = _base['suite'] == 'CV'
cv_cache = {k: _base[k][_cv] for k in ('sim_id', 'logM', 'p_dmo', 'p_t', 'p_g', 'm_t', 'm_g', 'e1_d', 'e2_d', 'e1_st', 'e2_st', 'e1_sg', 'e2_sg')}
cv_cache['r_pix'] = _base['r_pix']
print('CV halos:', cv_cache['logM'].size, 'across', len(np.unique(cv_cache['sim_id'])), 'sims')
print('logM:', cv_cache['logM'].min().round(2), '–', cv_cache['logM'].max().round(2),
      '  median', np.median(cv_cache['logM']).round(2))

## 2. Feature definitions

Per (B, 3, H, W) batch of physical patches we compute scalar observables.
All masses are sums over patch.  Fractions normalise out the M$_{200c}$ scaling:

- $M_{\rm DM}, M_{\rm gas}, M_\star$  — patch totals per channel
- $f_\star = M_\star / (M_{\rm DM} + M_{\rm gas} + M_\star)$, $f_{\rm gas}$ analogous
- compactness$_{\star,{\rm gas}}$  = mass within inner 8 radial bins / total
- $q_\star, q_{\rm gas}, q_{\rm DM,hydro}$, $q_{\rm DMO}$ — full-patch axis ratios
- $q_{\rm DM,hydro}/q_{\rm DMO}$  — back-reaction ratio

In [ ]:
INNER_BINS = 8  # ~ inner 0.25 R_patch ≈ 1.6 Mpc/h

def _radial_profiles(maps_2d, n_bins=N_RBINS):
    """(N, H, W) -> (N, n_bins)"""
    out = np.empty((len(maps_2d), n_bins), dtype=np.float64)
    for i, m in enumerate(maps_2d):
        _, prof = radial_profile(m, n_bins=n_bins)
        out[i] = prof
    return out

def _quadrupole_q(maps_2d, threshold=0.0, min_pixels=5):
    """(N, H, W) -> q (axis ratio in [0,1]); NaN for degenerate maps."""
    N, H, W = maps_2d.shape
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)
    q_out = np.full(N, np.nan)
    for i in range(N):
        w = np.maximum(maps_2d[i].astype(np.float64) - threshold, 0.0)
        tot = w.sum()
        if tot < 1e-30 or int((w > 0).sum()) < min_pixels:
            continue
        x_c = (xx * w).sum() / tot
        y_c = (yy * w).sum() / tot
        dx, dy = xx - x_c, yy - y_c
        Qxx = (dx * dx * w).sum() / tot
        Qyy = (dy * dy * w).sum() / tot
        Qxy = (dx * dy * w).sum() / tot
        evals = np.linalg.eigvalsh([[Qxx, Qxy], [Qxy, Qyy]])
        if evals[1] < 1e-30 or evals[0] < 0:
            continue
        q_out[i] = float(np.sqrt(evals[0] / evals[1]))
    return q_out

FEATURE_NAMES = [
    'M_DM', 'M_gas', 'M_star',
    'f_gas', 'f_star',
    'compact_star', 'compact_gas',
    'q_DM', 'q_gas', 'q_star',
    'q_DM_over_q_DMO',
]

def compute_features(patches_BCHW: np.ndarray, dmo_BHW: np.ndarray) -> dict:
    """Scalar features for a batch of (B, 3, H, W) hydro patches.
    `dmo_BHW` is the matched DMO patch per row, used for q_DMO.
    """
    dm, gas, star = patches_BCHW[:, 0], patches_BCHW[:, 1], patches_BCHW[:, 2]
    M_DM   = dm.sum(axis=(1, 2))
    M_gas  = gas.sum(axis=(1, 2))
    M_star = star.sum(axis=(1, 2))
    M_tot  = M_DM + M_gas + M_star + 1e-30

    # Compactness from radial profiles
    p_star = _radial_profiles(star)
    p_gas  = _radial_profiles(gas)
    # Profile is mean per annulus; need annulus pixel counts to get enclosed mass.
    # Use simple ratio of inner-bin profile sum to total profile sum (geometric weight cancels).
    yy, xx = np.mgrid[0:PATCH_PIX, 0:PATCH_PIX] - PATCH_PIX / 2
    rr = np.sqrt(xx ** 2 + yy ** 2)
    bins = np.linspace(0, PATCH_PIX / 2, N_RBINS + 1)
    counts = np.array([((rr >= bins[k]) & (rr < bins[k + 1])).sum() for k in range(N_RBINS)], dtype=np.float64)
    enc_star = (p_star * counts).cumsum(axis=1)
    enc_gas  = (p_gas  * counts).cumsum(axis=1)
    compact_star = enc_star[:, INNER_BINS - 1] / (enc_star[:, -1] + 1e-30)
    compact_gas  = enc_gas[:,  INNER_BINS - 1] / (enc_gas[:,  -1] + 1e-30)

    q_DM   = _quadrupole_q(dm)
    q_gas  = _quadrupole_q(gas)
    q_star = _quadrupole_q(star)
    q_DMO  = _quadrupole_q(dmo_BHW)
    q_back = q_DM / (q_DMO + 1e-30)

    return dict(
        M_DM=M_DM, M_gas=M_gas, M_star=M_star,
        f_gas=M_gas / M_tot, f_star=M_star / M_tot,
        compact_star=compact_star, compact_gas=compact_gas,
        q_DM=q_DM, q_gas=q_gas, q_star=q_star,
        q_DM_over_q_DMO=q_back,
    )

def features_to_matrix(feat: dict) -> np.ndarray:
    return np.stack([feat[k] for k in FEATURE_NAMES], axis=-1)  # (B, F)

## 3. Truth-feature matrix (all CV halos, recomputed for consistency)

We could read totals from `cv_cache['m_t']` directly, but compactness and the
DMO-axis ratio are not in the base cache — easier to recompute from full-box
and halo cutouts.  This pass touches every CV sim once.

In [ ]:
def _load_sim_arrays(suite: str, sim_id: str):
    base = SUITE_ROOT / suite / sim_id / SNAP
    full = np.load(base / 'full_maps.npz')
    cat  = np.load(base / MASS_TAG / 'halo_catalog.npz')
    cuts = np.load(base / MASS_TAG / 'halo_cutouts.npz')
    gen  = np.load(base / MASS_TAG / MODEL_NAME / 'generated_halos.npz')['generated']
    return full, cat, cuts, gen

def _extract_patches(full_maps, centers_pix):
    """Truth patches (n, 3, H, W) and DMO patches (n, H, W) from full-box maps."""
    n = len(centers_pix)
    truth = np.zeros((n, 3, PATCH_PIX, PATCH_PIX), dtype=np.float32)
    dmo   = np.zeros((n, PATCH_PIX, PATCH_PIX), dtype=np.float32)
    half = PATCH_PIX // 2
    for i, (cx, cy) in enumerate(centers_pix):
        ix = (cx - half + np.arange(PATCH_PIX)) % N_PIX_FULL
        iy = (cy - half + np.arange(PATCH_PIX)) % N_PIX_FULL
        dmo[i] = full_maps['dmo_fullbox'][np.ix_(ix, iy)]
        for c in range(3):
            truth[i, c] = full_maps['truth_maps'][c][np.ix_(ix, iy)]
    return truth, dmo

def _centers_pix(centers_mpc):
    ppm = N_PIX_FULL / BOX_SIZE
    return (np.asarray(centers_mpc) * ppm).astype(np.int64) % N_PIX_FULL

def build_truth_features(force=False):
    out = {n: [] for n in FEATURE_NAMES}
    out['sim_id'] = []; out['logM'] = []
    cv_dirs = sorted((SUITE_ROOT / 'CV').iterdir())
    for sd in tqdm(cv_dirs, desc='CV truth'):
        full, cat, _cuts, _gen = _load_sim_arrays('CV', sd.name)
        cpx = _centers_pix(cat['centers'])
        truth, dmo = _extract_patches(full, cpx)
        feat = compute_features(truth, dmo)
        for n in FEATURE_NAMES:
            out[n].append(feat[n])
        out['sim_id'].extend([sd.name] * len(cpx))
        out['logM'].extend(np.log10(np.asarray(cat['masses'], dtype=np.float64)).tolist())
    out = {k: (np.asarray(v) if k in ('sim_id',) else np.concatenate(v) if isinstance(v, list) and len(v) and hasattr(v[0], 'shape') else np.asarray(v)) for k, v in out.items()}
    return out

TRUTH_CACHE = CACHE_DIR / f'proj4_truth_features_{MODEL_NAME}.npz'
if TRUTH_CACHE.exists():
    print('Loading', TRUTH_CACHE)
    z = np.load(TRUTH_CACHE, allow_pickle=False)
    truth_feat = {k: z[k] for k in z.files}
else:
    truth_feat = build_truth_features()
    np.savez_compressed(TRUTH_CACHE, **truth_feat)
    print('Wrote', TRUTH_CACHE)

print('truth halos:', len(truth_feat['logM']))
print('feature keys:', [k for k in truth_feat if k in FEATURE_NAMES])

## 4. Stratified halo selection

5 logM quintiles × 10 halos each = 50.  We pick from the truth-feature table
so the same halos can be referenced for the K=64 generation.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
logM = truth_feat['logM']
edges = np.quantile(logM, np.linspace(0, 1, N_BINS_M + 1))
edges[0] -= 1e-6; edges[-1] += 1e-6
bin_idx = np.digitize(logM, edges) - 1
selected = []
for b in range(N_BINS_M):
    cand = np.where(bin_idx == b)[0]
    pick = rng.choice(cand, size=min(HALOS_PER_BIN, len(cand)), replace=False)
    selected.append(pick)
selected = np.concatenate(selected)
print('Selected', len(selected), 'halos across', N_BINS_M, 'mass bins')
print('Mass-bin edges (logM):', np.round(edges, 3))
for b in range(N_BINS_M):
    m = bin_idx[selected] == b
    print(f'  bin {b}: n={m.sum()}, logM in [{logM[selected][m].min():.2f}, {logM[selected][m].max():.2f}]')

selected_records = pd.DataFrame({
    'sim_id': truth_feat['sim_id'][selected],
    'row_in_truth': selected,
    'logM': logM[selected],
    'mass_bin': bin_idx[selected],
})
print('\nselection (head):')
print(selected_records.head())

## 5. Generate $K{=}64$ BIND samples per selected halo

Loads the same checkpoint and norm stats the existing test-suite uses
([test_suite/runner.py](test_suite/runner.py)), then loops halo-by-halo with
$K{=}64$ replicates batched at `BATCH_SIZE=16`.  Cached to
[analysis_physics_cache/](analysis_physics_cache/) so re-running the notebook
is free.

In [ ]:
def load_model():
    norm_stats = NormStats.load(NORM_STATS)
    lit = FlowMatchingLit.load_from_checkpoint(CKPT, map_location=device)
    lit.eval(); lit.to(device)
    return norm_stats, lit.fm

def _generate_K_for_halo(fm, norm_stats, hc_dict, sim_params, K, n_steps, batch_size):
    """Replicate one (cond, large_scale, params) K times and run K samples.
    Returns physical (K, 3, H, W) array."""
    cond_n, ls_n, p_n = normalize_cutout(hc_dict, norm_stats, sim_params)
    cond_t = torch.from_numpy(cond_n[None].astype(np.float32))
    ls_t   = torch.from_numpy(ls_n[None].astype(np.float32))
    p_t    = torch.from_numpy(p_n[None].astype(np.float32))

    out = []
    from contextlib import nullcontext
    amp_ctx = (torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
               if USE_AMP and device.type == 'cuda' else nullcontext())
    with torch.no_grad():
        for start in range(0, K, batch_size):
            B = min(batch_size, K - start)
            cb = cond_t.repeat(B, 1, 1, 1).to(device)
            lb = ls_t.repeat(B, 1, 1, 1).to(device)
            pb = p_t.repeat(B, 1).to(device)
            with amp_ctx:
                gen = fm.sample(cb, lb, pb, n_steps=n_steps)
            out.append(gen.float().cpu().numpy().astype(np.float32))
    gen_K = np.concatenate(out, axis=0)            # (K, C_model, H, W)
    return _denormalize_to_physical(gen_K, norm_stats)  # (K, 3, H, W)

def build_proj4_cache(force=False):
    if PROJ4_CACHE.exists() and not force:
        print('Loading', PROJ4_CACHE)
        z = np.load(PROJ4_CACHE, allow_pickle=False)
        return {k: z[k] for k in z.files}

    norm_stats, fm = load_model()

    # Group selected halos by sim so we open each sim once.
    by_sim = selected_records.groupby('sim_id', sort=False)

    feat_per_halo = []                # list of (K, F)
    halo_meta = []                    # list of dicts
    for sim_id, group in tqdm(by_sim, desc='Generating K samples'):
        full, cat, cuts, _gen = _load_sim_arrays('CV', sim_id)
        sim_params = np.asarray(cat['params'][0], dtype=np.float32)
        cpx = _centers_pix(cat['centers'])

        # Map row_in_truth -> halo index within this sim.
        # truth_feat rows are concatenated in CV-sim order; we know the sim_id per row.
        sim_rows_in_truth = np.where(truth_feat['sim_id'] == sim_id)[0]
        # Within the sim, halos are stored in the order in halo_catalog.
        # That is exactly the same order used to build truth_feat above.
        for _, rec in group.iterrows():
            local_idx = int(np.where(sim_rows_in_truth == rec['row_in_truth'])[0][0])
            hc_dict = {
                'condition':   cuts['condition'][local_idx],
                'large_scale': cuts['large_scale'][local_idx],
            }
            patches_K = _generate_K_for_halo(
                fm, norm_stats, hc_dict, sim_params,
                K=K_SAMPLES, n_steps=N_STEPS, batch_size=BATCH_SIZE,
            )
            cx, cy = cpx[local_idx]
            half = PATCH_PIX // 2
            ix = (cx - half + np.arange(PATCH_PIX)) % N_PIX_FULL
            iy = (cy - half + np.arange(PATCH_PIX)) % N_PIX_FULL
            dmo_patch = full['dmo_fullbox'][np.ix_(ix, iy)].astype(np.float32)
            dmo_K = np.broadcast_to(dmo_patch, (K_SAMPLES, PATCH_PIX, PATCH_PIX))
            feat = compute_features(patches_K, dmo_K)
            feat_per_halo.append(features_to_matrix(feat))
            halo_meta.append(dict(
                sim_id=sim_id, row_in_truth=int(rec['row_in_truth']),
                logM=float(rec['logM']), mass_bin=int(rec['mass_bin']),
            ))

    feat_arr = np.stack(feat_per_halo, axis=0)  # (n_halos, K, F)
    out = dict(
        features=feat_arr.astype(np.float32),
        feature_names=np.asarray(FEATURE_NAMES),
        sim_id=np.asarray([h['sim_id'] for h in halo_meta]),
        row_in_truth=np.asarray([h['row_in_truth'] for h in halo_meta]),
        logM=np.asarray([h['logM'] for h in halo_meta]),
        mass_bin=np.asarray([h['mass_bin'] for h in halo_meta]),
        K=np.asarray(K_SAMPLES),
        n_steps=np.asarray(N_STEPS),
    )
    np.savez_compressed(PROJ4_CACHE, **out)
    print('Wrote', PROJ4_CACHE)
    return out

# Run (will take ~30–60 min on A100 the first time).
proj4 = build_proj4_cache()
print('features:', proj4['features'].shape, '(n_halos, K, F)')

## 6. Variance decomposition

We compute, per mass bin:

- $\Sigma^{\rm truth}_{\rm bin}$  — covariance across CV truth halos in the bin
- $\Sigma^{\rm BIND, K=1}_{\rm bin}$  — covariance across BIND single-sample (one
  random sample per halo from the K=64 stack) within the selected subset
- $\langle \Sigma_{\rm intra} \rangle_{\rm bin}$  — mean intra-halo covariance
  over the K=64 stack
- $\Sigma^{\rm BIND, total}_{\rm bin} = \Sigma_{\rm K=1} + \langle \Sigma_{\rm intra} \rangle$

In [ ]:
def truth_matrix(bin_id):
    rows = (np.digitize(truth_feat['logM'], edges) - 1) == bin_id
    return np.stack([truth_feat[k][rows] for k in FEATURE_NAMES], axis=-1)

def proj4_intra_cov(features_NKF):
    """Mean of per-halo K-sample covariances. Returns (F, F)."""
    N, K, F = features_NKF.shape
    covs = np.zeros((N, F, F))
    for i in range(N):
        x = features_NKF[i]
        finite = np.isfinite(x).all(axis=1)
        x = x[finite]
        if len(x) >= 2:
            covs[i] = np.cov(x.T, ddof=1)
    return covs.mean(axis=0)

def safe_cov(arr_NF):
    finite = np.isfinite(arr_NF).all(axis=1)
    return np.cov(arr_NF[finite].T, ddof=1)

def cov_to_corr(C):
    d = np.sqrt(np.clip(np.diag(C), 1e-30, None))
    return C / np.outer(d, d)

rng_pick = np.random.default_rng(RNG_SEED + 1)
F = len(FEATURE_NAMES)
results = {}
for b in range(N_BINS_M):
    bm = proj4['mass_bin'] == b
    feats_bin = proj4['features'][bm]               # (n_halos_bin, K, F)
    if len(feats_bin) == 0:
        continue
    # Single-sample-per-halo across-halo covariance
    k_pick = rng_pick.integers(0, K_SAMPLES, size=len(feats_bin))
    feats_K1 = feats_bin[np.arange(len(feats_bin)), k_pick]  # (n_halos_bin, F)
    Sigma_K1    = safe_cov(feats_K1)
    Sigma_intra = proj4_intra_cov(feats_bin)
    Sigma_total = Sigma_K1 + Sigma_intra
    Sigma_truth = safe_cov(truth_matrix(b))
    results[b] = dict(
        truth=Sigma_truth, K1=Sigma_K1, intra=Sigma_intra, total=Sigma_total,
        n_truth=int(((np.digitize(truth_feat['logM'], edges) - 1) == b).sum()),
        n_bind=int(bm.sum()),
    )
    print(f'bin {b}: n_truth={results[b]["n_truth"]}, n_bind={results[b]["n_bind"]}')

## 7. Pre-registered tests

In [ ]:
from scipy.stats import ks_2samp

def feat_index(name): return FEATURE_NAMES.index(name)

rows = []
for b in range(N_BINS_M):
    if b not in results: continue
    R = results[b]
    bm = proj4['mass_bin'] == b
    feats_bin = proj4['features'][bm]
    truth_bin_M = truth_matrix(b)

    # Test 1: KS distance for f_star
    i_fs = feat_index('f_star')
    truth_fs = truth_bin_M[:, i_fs]; truth_fs = truth_fs[np.isfinite(truth_fs)]
    bind_fs  = feats_bin[:, :, i_fs].ravel(); bind_fs = bind_fs[np.isfinite(bind_fs)]
    ks = ks_2samp(truth_fs, bind_fs).statistic

    # Test 2: ratio of variances for f_star
    var_truth = R['truth'][i_fs, i_fs]
    var_total = R['total'][i_fs, i_fs]
    var_ratio = var_total / max(var_truth, 1e-30)

    # Test 3: corr(f_star, f_gas) — sign + |delta|
    i_fg = feat_index('f_gas')
    cor_truth = cov_to_corr(R['truth'])[i_fs, i_fg]
    cor_total = cov_to_corr(R['total'])[i_fs, i_fg]

    # Test 4: corr(q_DM/q_DMO, M_star)
    i_qb = feat_index('q_DM_over_q_DMO'); i_ms = feat_index('M_star')
    cor_truth_4 = cov_to_corr(R['truth'])[i_qb, i_ms]
    cor_total_4 = cov_to_corr(R['total'])[i_qb, i_ms]

    rows.append(dict(
        bin=b,
        ks_fstar=ks,
        var_ratio_fstar=var_ratio,
        cor_fstar_fgas_truth=cor_truth,
        cor_fstar_fgas_bind=cor_total,
        cor_dmback_mstar_truth=cor_truth_4,
        cor_dmback_mstar_bind=cor_total_4,
        pass_test1=ks < 0.10,
        pass_test2=(0.5 <= var_ratio <= 2.0),
        pass_test3=(np.sign(cor_truth) == np.sign(cor_total)) and (abs(cor_truth - cor_total) < 0.3),
        pass_test4=np.sign(cor_truth_4) == np.sign(cor_total_4),
    ))
tests_df = pd.DataFrame(rows)
tests_df.round(3)

## 8. Diagnostic figures

**Figure A.** Side-by-side correlation matrices (truth vs BIND-total) per
mass bin — visual check that off-diagonal *structure* matches.

**Figure B.** $f_\star$ vs $f_{\rm gas}$ scatter for one mass bin: truth halos
(blue), BIND across-halo K=1 (orange), and BIND intra-halo cloud for one
example halo (gray cloud).  The intra-halo cloud should *match the slope* of
the truth scatter if BIND learned the budget physics.

In [ ]:
# Figure A: correlation-matrix grid
active_bins = [b for b in range(N_BINS_M) if b in results]
fig, axes = plt.subplots(2, len(active_bins), figsize=(3.0 * len(active_bins), 6.0))
for j, b in enumerate(active_bins):
    R = results[b]
    for r, (lab, C) in enumerate([('truth', R['truth']), ('BIND total', R['total'])]):
        ax = axes[r, j]
        im = ax.imshow(cov_to_corr(C), vmin=-1, vmax=1, cmap='RdBu_r')
        ax.set_xticks(range(F)); ax.set_yticks(range(F))
        ax.set_xticklabels(FEATURE_NAMES, rotation=90, fontsize=7)
        ax.set_yticklabels(FEATURE_NAMES, fontsize=7)
        ax.set_title(f'bin {b} ({lab})  n={R["n_truth"] if lab=="truth" else R["n_bind"]}')
fig.colorbar(im, ax=axes, fraction=0.02, label='Pearson r')
fig.suptitle('Per-mass-bin feature correlation: CAMELS truth vs BIND2 (intra+inter)')
fig.savefig(FIG_DIR / 'proj4_corr_matrices.png', dpi=150)
plt.show()

In [ ]:
# Figure B: f_star vs f_gas scatter for the median mass bin
b_show = active_bins[len(active_bins) // 2]
i_fs, i_fg = feat_index('f_star'), feat_index('f_gas')
T = truth_matrix(b_show)
bm = proj4['mass_bin'] == b_show
feats_bin = proj4['features'][bm]
k_pick = rng_pick.integers(0, K_SAMPLES, size=len(feats_bin))
K1 = feats_bin[np.arange(len(feats_bin)), k_pick]
example_h = feats_bin[0]  # first halo's K cloud

fig, ax = plt.subplots(figsize=(5.0, 4.0))
ax.scatter(T[:, i_fs], T[:, i_fg], s=14, alpha=0.5, label=f'truth (n={len(T)})', color='tab:blue')
ax.scatter(K1[:, i_fs], K1[:, i_fg], s=22, marker='x', label=f'BIND K=1 (n={len(K1)})', color='tab:orange')
ax.scatter(example_h[:, i_fs], example_h[:, i_fg], s=8, alpha=0.4, color='gray',
           label='BIND intra-halo cloud (one halo, K=64)')
ax.set_xlabel(r'$f_\star$'); ax.set_ylabel(r'$f_{\rm gas}$')
ax.set_title(f'Mass bin {b_show}: BIND vs truth budget coupling')
ax.legend(loc='best', fontsize=8)
fig.savefig(FIG_DIR / 'proj4_fstar_fgas_scatter.png', dpi=150)
plt.show()

## 9. Interpretation cheatsheet

- **Test 1 fails** → BIND's marginal $f_\star$ distribution doesn't match
  truth.  Calibration problem before anything else; resolve before reading
  off-diagonals.
- **Test 2 fails high (>2×)** → BIND is over-dispersed; generative noise
  exceeds physical scatter.
- **Test 2 fails low (<0.5×)** → BIND is under-dispersed; mode collapse or
  near-deterministic mapping.
- **Tests 1+2 pass but Test 3 fails** → diagnostic is meaningful and BIND is
  *interpolating, not learning physics* — variances are right but the
  $M_\star$/$M_{\rm gas}$ coupling is missing.
- **Test 4 fails** → DM back-reaction is uncorrelated with stellar growth in
  the model; another sign of missing baryonic-budget coupling.
- **All tests pass** → BIND captures budget physics at fiducial parameters.
  Next step: re-run at one or two non-fiducial 1P points (e.g., extreme
  $A_{\rm SN1}$ and $A_{\rm AGN1}$) to test calibration off fiducial.